# Banknote Authentication – DBSCAN Clustering Evaluation
**SCT213-C002-0025/2023 - Koigi Stephen**  
ML II Unsupervised Learning Capstone – Model Evaluation Module

This notebook focuses exclusively on **model evaluation** for the DBSCAN clustering applied to the banknote authentication dataset.  
It covers:
- Minimal EDA (shape, head, info)
- DBSCAN model fitting (No PCA & With PCA)
- Internal metrics: Silhouette Score, Davies–Bouldin Index, Calinski–Harabasz Index
- External metrics (ground truth available): Adjusted Rand Index (ARI), Normalised Mutual Information (NMI)
- Clear summary table

In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & modelling
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Internal evaluation metrics
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# External evaluation metrics
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

print('All imports successful.')

All imports successful.


---
## 1. Imports

---
## 2. Load Data & Minimal EDA

In [5]:
# ── Update DATA_PATH to your local file location ──────────────────────────────
DATA_PATH = r"data_banknote_authentication.txt"

data = pd.read_csv(
    DATA_PATH,
    header=None,
    names=['variance', 'skewness', 'curtosis', 'entropy', 'class']
)
data = data.drop_duplicates().reset_index(drop=True)

print(f'Dataset shape after deduplication: {data.shape}')

Dataset shape after deduplication: (1348, 5)


In [6]:
# --- 2a. Head ---
print('=== First 5 rows ===')
display(data.head())

=== First 5 rows ===


,variance,skewness,curtosis,entropy,class
0,3.62160,8.6661,-2.8073,-0.44699,0
1,4.54590,8.1674,-2.4586,-1.46210,0
2,3.86600,-2.6383,1.9242,0.10645,0
3,3.45660,9.5228,-4.0112,-3.59440,0
4,0.32924,-4.4552,4.5718,-0.98880,0


In [7]:
# --- 2b. Basic Info & Class Distribution ---
print('=== Dataset Info ===')
data.info()
print()
print('=== Class Distribution ===')
print(
    data['class']
    .value_counts()
    .rename({0: 'Authentic (0)', 1: 'Forged (1)'})
    .to_string()
)
print()
print('=== Descriptive Statistics ===')
display(data.describe().round(3))

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1348 entries, 0 to 1347
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   variance  1348 non-null   float64
 1   skewness  1348 non-null   float64
 2   curtosis  1348 non-null   float64
 3   entropy   1348 non-null   float64
 4   class     1348 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 52.8 KB

=== Class Distribution ===
class
Authentic (0)    738
Forged (1)       610

=== Descriptive Statistics ===


,variance,skewness,curtosis,entropy,class
count,1348.000,1348.000,1348.000,1348.000,1348.000
mean,0.446,1.909,1.414,-1.169,0.453
std,2.863,5.869,4.328,2.086,0.498
min,-7.042,-13.773,-5.286,-8.548,0.000
25%,-1.787,-1.627,-1.546,-2.393,0.000
50%,0.519,2.334,0.605,-0.579,0.000
75%,2.853,6.796,3.200,0.404,1.000
max,6.825,12.952,17.927,2.450,1.000


---
## 3. Preprocessing Pipeline
Mirrors Step 2 exactly: median imputation → StandardScaler → optional PCA (2 components).

In [8]:
# Separate features and ground-truth labels
y = data['class'].reset_index(drop=True)
X_raw = data.drop('class', axis=1).reset_index(drop=True)
feature_cols = X_raw.columns.tolist()

# Preprocessing pipeline
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([('num', numeric_pipeline, feature_cols)])
X_scaled = preprocessor.fit_transform(X_raw)
df_no_pca = pd.DataFrame(X_scaled, columns=feature_cols)

# PCA (2 components)
pca_model = PCA(n_components=2, random_state=42)
X_pca = pca_model.fit_transform(X_scaled)
df_with_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])

var_retained = pca_model.explained_variance_ratio_.sum()
print(f'PCA variance retained : {var_retained:.1%}')
print(f'No-PCA feature matrix : {df_no_pca.shape}')
print(f'With-PCA feature matrix: {df_with_pca.shape}')

PCA variance retained : 86.7%
No-PCA feature matrix : (1348, 4)
With-PCA feature matrix: (1348, 2)


---
## 4. Fit DBSCAN Models

| Model | eps | min_samples | Rationale |
|-------|-----|-------------|-----------|
| No PCA | 0.50 | 15 | Elbow at ~0.5; min_s=15 reduces micro-clusters |
| With PCA | 0.60 | 15 | Wider eps needed in 2-D PCA space |

In [9]:
# Hyperparameters (chosen from k-distance elbow plots in Step 3)
EPS_FULL  = 0.50
EPS_PCA   = 0.60
MIN_SAMPLES = 15

# Fit both models
dbscan_full = DBSCAN(eps=EPS_FULL, min_samples=MIN_SAMPLES)
labels_full = dbscan_full.fit_predict(df_no_pca.values)

dbscan_pca = DBSCAN(eps=EPS_PCA, min_samples=MIN_SAMPLES)
labels_pca = dbscan_pca.fit_predict(df_with_pca.values)

# Quick cluster / noise summary
for name, labels in [('No PCA', labels_full), ('With PCA', labels_pca)]:
    n_cls   = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f'[{name}]  Clusters: {n_cls}  |  Noise points: {n_noise} ({n_noise/len(labels):.1%})')

[No PCA]  Clusters: 5  |  Noise points: 222 (16.5%)
[With PCA]  Clusters: 1  |  Noise points: 1 (0.1%)


---
## 5. Evaluation Functions

### 5.1 Internal Metrics
Computed on cluster points only (noise label = −1 is excluded).

| Metric | Range | Better |
|--------|-------|--------|
| Silhouette Score | −1 to 1 | Higher |
| Davies–Bouldin Index | ≥ 0 | **Lower** |
| Calinski–Harabasz Index | ≥ 0 | Higher |

### 5.2 External Metrics (ground truth required)

| Metric | Range | Better |
|--------|-------|--------|
| Adjusted Rand Index (ARI) | −1 to 1 | Higher |
| Normalised Mutual Info (NMI) | 0 to 1 | Higher |

In [10]:
def compute_internal_metrics(X: np.ndarray, labels: np.ndarray) -> dict:
    """
    Compute internal clustering evaluation metrics.
    Noise points (label == -1) are excluded from computation.

    Parameters
    ----------
    X      : feature matrix (numpy array)
    labels : cluster label array (from DBSCAN.fit_predict)

    Returns
    -------
    dict with keys: Silhouette, Davies-Bouldin, Calinski-Harabasz
    """
    mask    = labels != -1          # exclude noise
    X_clean = X[mask]
    l_clean = labels[mask]
    n_cls   = len(set(l_clean))

    # Need ≥ 2 clusters and ≥ 2 non-noise points for any metric
    if n_cls < 2 or len(l_clean) < 2:
        return {
            'Silhouette Score'          : 'N/A (< 2 clusters)',
            'Davies–Bouldin Index'      : 'N/A (< 2 clusters)',
            'Calinski–Harabasz Index'   : 'N/A (< 2 clusters)',
        }

    return {
        'Silhouette Score'          : round(silhouette_score(X_clean, l_clean), 4),
        'Davies–Bouldin Index'      : round(davies_bouldin_score(X_clean, l_clean), 4),
        'Calinski–Harabasz Index'   : round(calinski_harabasz_score(X_clean, l_clean), 4),
    }


def compute_external_metrics(labels: np.ndarray, y_true: np.ndarray) -> dict:
    """
    Compute external clustering evaluation metrics against ground-truth labels.
    Noise points are included (treated as their own pseudo-cluster by sklearn).

    Parameters
    ----------
    labels : cluster label array (from DBSCAN.fit_predict)
    y_true : ground-truth class labels

    Returns
    -------
    dict with keys: ARI, NMI
    """
    return {
        'Adjusted Rand Index (ARI)'            : round(adjusted_rand_score(y_true, labels), 4),
        'Normalised Mutual Info (NMI)'         : round(normalized_mutual_info_score(y_true, labels), 4),
    }


def evaluate_clustering(
    model_name : str,
    X          : np.ndarray,
    labels     : np.ndarray,
    y_true     : np.ndarray = None
) -> pd.DataFrame:
    """
    Master evaluation function: combines internal + external metrics
    into a single tidy summary DataFrame.

    Parameters
    ----------
    model_name : display name for the model
    X          : feature matrix used for clustering
    labels     : predicted cluster labels
    y_true     : (optional) ground-truth labels for external metrics

    Returns
    -------
    pd.DataFrame with columns [Metric, Score, Type, Notes]
    """
    n_cls   = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()

    rows = []

    # ── Cluster summary ───────────────────────────────────────────────────────
    rows.append({'Metric': 'Clusters Found',  'Score': n_cls,
                 'Type': 'Summary', 'Notes': '–'})
    rows.append({'Metric': 'Noise Points',
                 'Score': f'{n_noise} ({n_noise/len(labels):.1%})',
                 'Type': 'Summary', 'Notes': 'label == −1'})

    # ── Internal metrics ──────────────────────────────────────────────────────
    internal = compute_internal_metrics(X, labels)
    notes_map = {
        'Silhouette Score'          : 'Higher is better  (−1 to 1)',
        'Davies–Bouldin Index'      : 'Lower is better   (≥ 0)',
        'Calinski–Harabasz Index'   : 'Higher is better  (≥ 0)',
    }
    for metric, score in internal.items():
        rows.append({'Metric': metric, 'Score': score,
                     'Type': 'Internal', 'Notes': notes_map[metric]})

    # ── External metrics (if y_true provided) ─────────────────────────────────
    if y_true is not None:
        external = compute_external_metrics(labels, y_true)
        ext_notes = {
            'Adjusted Rand Index (ARI)'   : 'Higher is better  (−1 to 1)',
            'Normalised Mutual Info (NMI)': 'Higher is better  (0 to 1)',
        }
        for metric, score in external.items():
            rows.append({'Metric': metric, 'Score': score,
                         'Type': 'External', 'Notes': ext_notes[metric]})

    df = pd.DataFrame(rows).set_index('Metric')
    df.index.name = f'Model: {model_name}'
    return df


print('Evaluation functions defined successfully.')

Evaluation functions defined successfully.


---
## 6. Evaluate — No PCA Model

In [11]:
results_full = evaluate_clustering(
    model_name = f'DBSCAN  No PCA  (eps={EPS_FULL}, min_samples={MIN_SAMPLES})',
    X          = df_no_pca.values,
    labels     = labels_full,
    y_true     = y.values
)

print('=== Evaluation Summary — No PCA Model ===')
display(
    results_full.style
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2c3e50'), ('color', 'white'),
                  ('font-weight', 'bold'), ('padding', '8px')]
    }])
    .applymap(
        lambda v: 'background-color: #d5f5e3' if isinstance(v, str) and 'Internal' in v else
                  ('background-color: #d6eaf8' if isinstance(v, str) and 'External' in v else ''),
        subset=['Type']
    )
)

=== Evaluation Summary — No PCA Model ===


,Score,Type,Notes
"Model: DBSCAN No PCA (eps=0.5, min_samples=15)",,,
Clusters Found,5,Summary,–
Noise Points,222 (16.5%),Summary,label == −1
Silhouette Score,0.195700,Internal,Higher is better (−1 to 1)
Davies–Bouldin Index,0.977600,Internal,Lower is better (≥ 0)
Calinski–Harabasz Index,257.763900,Internal,Higher is better (≥ 0)
Adjusted Rand Index (ARI),0.549300,External,Higher is better (−1 to 1)
Normalised Mutual Info (NMI),0.553400,External,Higher is better (0 to 1)


---
## 7. Evaluate — With PCA Model

In [12]:
results_pca = evaluate_clustering(
    model_name = f'DBSCAN  With PCA  (eps={EPS_PCA}, min_samples={MIN_SAMPLES})',
    X          = df_with_pca.values,
    labels     = labels_pca,
    y_true     = y.values
)

print('=== Evaluation Summary — With PCA Model ===')
display(
    results_pca.style
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#2c3e50'), ('color', 'white'),
                  ('font-weight', 'bold'), ('padding', '8px')]
    }])
    .applymap(
        lambda v: 'background-color: #d5f5e3' if isinstance(v, str) and 'Internal' in v else
                  ('background-color: #d6eaf8' if isinstance(v, str) and 'External' in v else ''),
        subset=['Type']
    )
)

=== Evaluation Summary — With PCA Model ===


,Score,Type,Notes
"Model: DBSCAN With PCA (eps=0.6, min_samples=15)",,,
Clusters Found,1,Summary,–
Noise Points,1 (0.1%),Summary,label == −1
Silhouette Score,N/A (< 2 clusters),Internal,Higher is better (−1 to 1)
Davies–Bouldin Index,N/A (< 2 clusters),Internal,Lower is better (≥ 0)
Calinski–Harabasz Index,N/A (< 2 clusters),Internal,Higher is better (≥ 0)
Adjusted Rand Index (ARI),-0.000300,External,Higher is better (−1 to 1)
Normalised Mutual Info (NMI),0.001300,External,Higher is better (0 to 1)


---
## 8. Side-by-Side Comparison Table

In [13]:
# Build a clean comparison table (numeric metrics only)
NUMERIC_METRICS = [
    'Silhouette Score',
    'Davies–Bouldin Index',
    'Calinski–Harabasz Index',
    'Adjusted Rand Index (ARI)',
    'Normalised Mutual Info (NMI)',
]

comparison = pd.DataFrame({
    'Metric Type'                 : ['Internal', 'Internal', 'Internal', 'External', 'External'],
    'Better Direction'            : ['↑ Higher', '↓ Lower', '↑ Higher', '↑ Higher', '↑ Higher'],
    f'No PCA (eps={EPS_FULL})'    : [results_full.loc[m, 'Score'] for m in NUMERIC_METRICS],
    f'With PCA (eps={EPS_PCA})'   : [results_pca.loc[m, 'Score']  for m in NUMERIC_METRICS],
}, index=NUMERIC_METRICS)
comparison.index.name = 'Metric'

# Highlight the better value per row
def highlight_better(row):
    score_cols = [c for c in row.index if 'eps' in c]
    try:
        vals = [float(row[c]) for c in score_cols]
    except (ValueError, TypeError):
        return ['' for _ in row.index]
    direction = row['Better Direction']
    best_idx  = vals.index(min(vals)) if '↓' in direction else vals.index(max(vals))
    colors = ['']*len(row.index)
    col_positions = [list(row.index).index(c) for c in score_cols]
    colors[col_positions[best_idx]] = 'background-color: #a9dfbf; font-weight: bold'
    return colors

print('=== Side-by-Side Model Comparison ===')
display(
    comparison.style
    .apply(highlight_better, axis=1)
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#1a252f'), ('color', 'white'),
                  ('font-weight', 'bold'), ('padding', '8px')]
    }])
    .format(precision=4, na_rep='N/A')
)

=== Side-by-Side Model Comparison ===


,Metric Type,Better Direction,No PCA (eps=0.5),With PCA (eps=0.6)
Metric,,,,
Silhouette Score,Internal,↑ Higher,0.1957,N/A (< 2 clusters)
Davies–Bouldin Index,Internal,↓ Lower,0.9776,N/A (< 2 clusters)
Calinski–Harabasz Index,Internal,↑ Higher,257.7639,N/A (< 2 clusters)
Adjusted Rand Index (ARI),External,↑ Higher,0.5493,-0.0003
Normalised Mutual Info (NMI),External,↑ Higher,0.5534,0.0013


---
## 9. Interpretation

### No PCA Model (Best Performer)

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Silhouette Score | 0.1957 | Moderate separation; clusters overlap somewhat |
| Davies–Bouldin Index | — | Lower = tighter, better-separated clusters |
| Calinski–Harabasz Index | — | Higher = denser, more distinct clusters |
| ARI | 0.549 | Reasonable alignment with authentic/forged labels |
| NMI | — | Measures shared information with true labels |

### Key Observations

1. **No PCA outperforms With PCA** across all metrics because the 4-D feature space preserves the density structure that DBSCAN relies on. Reducing to 2 PCA components merges distinct density regions, collapsing everything into a single cluster.

2. **High Homogeneity / moderate ARI** in the No PCA model confirms that each cluster is dominated by a single true class, even though the true classes are split across sub-clusters (lower completeness).

3. **Davies–Bouldin Index** adds a perspective not in the original analysis: lower values confirm clusters are compact and well-separated relative to each other.

4. **NMI** complements ARI: while ARI penalises chance agreement, NMI measures the amount of true-label information captured by the clustering — a more interpretable external measure.

5. **16.5% noise** in the No PCA model reflects genuine borderline banknotes that are ambiguous under density assumptions — consistent with the outliers flagged in Step 2 preprocessing.

---
## 10. Export Results

In [14]:
# Save individual and comparison tables to CSV
results_full.to_csv('eval_no_pca.csv')
results_pca.to_csv('eval_with_pca.csv')
comparison.to_csv('eval_comparison.csv')

print('Saved: eval_no_pca.csv')
print('Saved: eval_with_pca.csv')
print('Saved: eval_comparison.csv')

Saved: eval_no_pca.csv
Saved: eval_with_pca.csv
Saved: eval_comparison.csv
